# Data Simulator

Generates synthetic training movies for `EventDetector`: binding, unbinding,
and movement (dipole) events composited onto real instrument background/noise
sampled from recorded buffer movies.

In [ ]:
import numpy as np
from abc import ABC, abstractmethod
from dataclasses import dataclass

from alex_area.movie_generator.buffer_movies import BufferMovie, load_buffer_movies

In [ ]:
MIN_FRAME_DIM_SIZE: int = 42  # smallest crop width/height sampled by gen_random_mov_stack

In [ ]:
B_MOV_X_MAX: int = 231
B_MOV_Y_MAX: int = 163

# Buffer movies capture real instrument background/noise with no particle events;
# indices 12+ correspond to TwoMP large-FoV buffer movies.
b_movs = load_buffer_movies()[12:]
BUFFER_MOVIES = [
    mov[:, :B_MOV_Y_MAX, :B_MOV_X_MAX] for mov in b_movs
]  # crop all buffer movies to a common (Y, X) footprint

In [ ]:
def gen_random_mov_stack() -> BufferMovie:
    """Sample a random crop from a randomly chosen buffer movie.

    Selects one of `BUFFER_MOVIES` at random, then crops it to a random
    (y_width, x_width) window (each dimension between `MIN_FRAME_DIM_SIZE`
    and the movie's full extent) placed at a random offset. Used to vary the
    field-of-view size and background content seen during training.

    Returns:
        The cropped buffer movie, shape (T, y_width, x_width).
    """
    rand_index = np.random.randint(0, len(BUFFER_MOVIES))
    mov: BufferMovie = BUFFER_MOVIES[rand_index]

    x_width = np.random.randint(MIN_FRAME_DIM_SIZE, B_MOV_X_MAX + 1)
    y_width = np.random.randint(MIN_FRAME_DIM_SIZE, B_MOV_Y_MAX + 1)

    x_max = B_MOV_X_MAX - x_width
    y_max = B_MOV_Y_MAX - y_width

    y_rand = np.random.randint(0, y_max + 1)
    x_rand = np.random.randint(0, x_max + 1)

    return mov[:, y_rand : y_rand + y_width, x_rand : x_rand + x_width]

In [ ]:
# TODO(WIP): placeholder synthetic movie generator -- revisit sampling approach.
rand_mov = np.random.randint(
    low=np.iinfo(np.uint16).min,
    high=np.iinfo(np.uint16).max + 1,
    size=(2500, MIN_FRAME_DIM_SIZE, MIN_FRAME_DIM_SIZE),
    dtype=np.uint16,
)

In [ ]:
class AbstractSimEvent(ABC):
    """Interface for a single simulated event placed into a training movie.

    Concrete events store their (x, y, i, c) coordinates as plain fields;
    this interface only constrains the values derived from them.
    """

    x: float
    y: float
    i: float
    c: float

    @property
    @abstractmethod
    def hot_px(self) -> tuple[int, int]:
        """Nearest integer (x, y) pixel, i.e. the heatmap peak location."""

    @property
    @abstractmethod
    def offset(self) -> tuple[float, float]:
        """Sub-pixel (dx, dy) offset of (x, y) from `hot_px`."""

    @abstractmethod
    def to_simple(self) -> list[list[float]]:
        """Flatten the event to one or more [x, y, i, c] records."""

In [ ]:
@dataclass
class BaseSimEvent(AbstractSimEvent):
    """Concrete single-point event: a binding or unbinding at (x, y, i, c).

    Attributes:
        x: Sub-pixel x-coordinate.
        y: Sub-pixel y-coordinate.
        i: Frame index (temporal coordinate).
        c: Contrast.
    """

    x: float
    y: float
    i: float
    c: float

    @property
    def hot_px(self) -> tuple[int, int]:
        return (np.round(self.x).astype(int), np.round(self.y).astype(int))

    @property
    def offset(self) -> tuple[float, float]:
        x_px, y_px = self.hot_px

        return (self.x - float(x_px), self.y - float(y_px))

    def to_simple(self) -> list[list[float]]:
        return [[self.x, self.y, self.i, self.c]]

In [ ]:
class BindingSimEvent(BaseSimEvent):
    """A particle landing (binding) event."""


class UnbindingSimEvent(BaseSimEvent):
    """A particle leaving (unbinding) event."""


@dataclass
class MovementSimEvent(BaseSimEvent):
    """A dipole movement: a linked unbinding-then-binding pair.

    Represents a particle moving from one location to another, modeled as an
    unbinding event and a binding event straddling the midpoint (x, y),
    separated by `distance` at angle `theta`. Both endpoints share the same
    intensity/frame index and contrast.

    Attributes:
        distance: Distance between the unbinding and binding endpoints.
        theta: Direction of travel, in radians.
    """

    distance: float
    theta: float

    def __post_init__(self) -> None:
        self.theta = (self.theta / (2 * np.pi)) - np.floor(
            self.theta / (2 * np.pi)
        )  # wrap to [0, 2*pi)

        self.dx: float = self.distance * np.cos(self.theta)
        self.dy: float = self.distance * np.sin(self.theta)

        self.unbinding: UnbindingSimEvent = UnbindingSimEvent(
            x=self.x - self.dx / 2, y=self.y - self.dy / 2, i=self.i, c=self.c
        )
        self.binding: BindingSimEvent = BindingSimEvent(
            x=self.x + self.dx / 2, y=self.y + self.dy / 2, i=self.i, c=self.c
        )

    def to_simple(self) -> list[list[float]]:
        """Flatten to the unbinding and binding endpoint records.

        Returns:
            A list of two [x, y, i, c] records: the unbinding endpoint
            followed by the binding endpoint.
        """
        return self.unbinding.to_simple() + self.binding.to_simple()

In [ ]:
def gen_events(
    n_binding: int,
    n_unbinding: int,
    n_movement: int,
) -> list[AbstractSimEvent]:
    """Sample a batch of random binding, unbinding, and movement events.

    Args:
        n_binding: Number of `BindingSimEvent`s to generate.
        n_unbinding: Number of `UnbindingSimEvent`s to generate.
        n_movement: Number of `MovementSimEvent`s to generate.

    Returns:
        The generated events, in no particular order.

    Raises:
        NotImplementedError: Sampling distributions for event position,
            timing, intensity, contrast, and movement distance/angle are not
            yet decided.
    """
    # TODO: implement once event sampling distributions are finalized.
    raise NotImplementedError